In [1]:
!pip install langchain openai faiss-cpu tiktoken pypdf
%pip install -qU langchain_community pypdf
!pip install chromadb
!pip install -U langchain langchain-community langchain-core langchain-text-splitters
!pip install langchain-anthropic
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.5 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2

In [2]:
#loading data
loader= PyPDFLoader("/kaggle/input/datasets/abdelrahmanramadan2/customer-support/FAQs.pdf")
docs= loader.load()

In [3]:
#split into chunks
splitter= RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

In [4]:
#convert to embedings + store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = Chroma.from_documents(chunks, embeddings)

/tmp/ipykernel_57/346121222.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
#Create Retriever
retriever = db.as_retriever(search_kwargs={"k": 4})
!pip install langchain-groq
from langchain_groq import ChatGroq
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GROQ_API_KEY  = user_secrets.get_secret("API KEY") 
llm = ChatGroq(
    model="llama-3.1-8b-instant",  
    api_key= GROQ_API_KEY 
)
prompt = ChatPromptTemplate.from_template("""
    You are an AI assistant that provides clear, structured responses **strictly using the provided knowledge base**.

    **User Question:** {query}

    **Relevant Information:**
    {retrieved_text}

    **Instructions:**
    - Answer **only** using the provided knowledge base.
    - Provide a **step-by-step list** if applicable.
    - Use **bullet points or numbered lists** where necessary.
    - **Elaborate** on each step, making sure it's clear and informative.
    - If no relevant information is found, state: 'I couldn't find enough details in my sources.'

    **Answer in this format:**
    1. **Step 1**: Explanation
    2. **Step 2**: Explanation
    3. **Step 3**: Explanation

    **Bullet Points Example:**
    * **Feature 1**: Explanation
    * **Feature 2**: Explanation

    **Final Answer:**
""")

chain = (
    {"retrieved_text": retriever, "query": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [11]:
#asking question
response = llm.invoke("How do I keep track of my returned orders?")


print(response.content)

Keeping track of returned orders is essential for any business or individual who sells products online or in-store. Here are some ways to keep track of returned orders:

1. **Use a Return Management System (RMS)**: Implement a RMS that allows you to track and manage returns, exchanges, and refunds. This can be a dedicated software or a feature within your existing e-commerce platform.
2. **Create a Return Policy**: Establish a clear return policy that outlines the process for returning products, including the timeframe, requirements, and any associated fees.
3. **Use a Return Tracking Sheet**: Keep a record of each return, including the order number, product details, reason for return, and current status.
4. **Implement a Returns Label**: Use a returns label that allows customers to easily print a return shipping label and track the return.
5. **Monitor Customer Communication**: Keep a record of all communication with customers regarding returns, including emails, phone calls, and in-p